# **1. Analisis Exploratorio de Datos Inicial**

## 1.1. Carga de datos

In [ ]:
import pandas as pd

archivos = [f"../data/yellow_tripdata_2026-0{i}.parquet" for i in range(1, 7)]

df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)

df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,NaN
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,NaN
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75,NaN
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,NaN
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22836525,2,2026-06-30 23:17:26,2026-06-30 23:28:30,NaN,1.22,NaN,None,230,90,0,...,0.00,0.5,0.00,0.0,1.0,23.23,NaN,NaN,0.75,HV0003
22836526,2,2026-06-30 23:10:28,2026-06-30 23:29:56,NaN,2.73,NaN,None,209,107,0,...,0.00,0.5,0.00,0.0,1.0,23.96,NaN,NaN,0.75,HV0003
22836527,2,2026-06-30 23:37:56,2026-06-30 23:45:09,NaN,1.24,NaN,None,74,41,0,...,0.00,0.5,0.00,0.0,1.0,10.78,NaN,NaN,0.00,HV0003
22836528,2,2026-06-30 23:53:02,2026-07-01 00:01:11,NaN,0.62,NaN,None,232,4,0,...,0.00,0.5,0.00,0.0,1.0,24.13,NaN,NaN,0.75,HV0003


Número de filas y columnas:

In [2]:
print("Numero de filas: ")
print(df.shape[0])
print("Numero de columnas: ")
print(df.shape[1])

Numero de filas: 
22836530
Numero de columnas: 
21


Tipos de datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22836530 entries, 0 to 22836529
Data columns (total 21 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee         

## 1.2. Analisis de valores faltantes

In [4]:
# Calcular valores nulos y porcentaje por columna
faltantes = pd.DataFrame({
    'Valores Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df)) * 100
})

# Ordenar de mayor a menor y mostrar solo las columnas que tienen nulos
faltantes = faltantes[faltantes['Valores Nulos'] > 0].sort_values(by='Valores Nulos', ascending=False)
faltantes

,Valores Nulos,Porcentaje (%)
request_source,21823350,95.563336
passenger_count,5825780,25.510793
RatecodeID,5825780,25.510793
store_and_fwd_flag,5825780,25.510793
congestion_surcharge,5825780,25.510793
Airport_fee,5825780,25.510793


## 1.3. Analisis de Variable Objetivo

In [ ]:
df['trip_duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

print(df['trip_duration'].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))



count    2.283653e+07
mean     1.778938e+01
std      2.534032e+01
min     -1.866667e+02
1%       0.000000e+00
5%       3.316667e+00
25%      8.300000e+00
50%      1.386667e+01
75%      2.210000e+01
95%      4.471667e+01
99%      7.386667e+01
max      9.983933e+03
Name: trip_duration, dtype: float64


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_filtrado = df[(df['trip_duration'] >= 1) & (df['trip_duration'] <= 120)]

# 4. Graficar Histograma y Boxplot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma / Densidad
sns.histplot(df_filtrado['trip_duration'], bins=50, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribución de trip_duration (Filtro: 1 a 120 min)')
axes[0].set_xlabel('Duración (Minutos)')
axes[0].set_ylabel('Frecuencia')

# Boxplot
sns.boxplot(x=df_filtrado['trip_duration'], ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot de trip_duration')
axes[1].set_xlabel('Duración (Minutos)')

plt.tight_layout()
plt.show()